È una baseline “memorize & match”:
1. prendo tutte le entità viste nel training
2. le cerco come substring nel testo del dev
3. se le trovo, le predico con la/le label associate
4. elimino overlap (tengo span più lunghi)
5. valuto con exact match.

Limiti grossi (che spiegano perché è davvero “baseline”)
- Case-sensitive: “Vitamin D” ≠ “vitamin d”
- Substring naive: match dentro parole, nessun token-boundary
- Zero generalizzazione: trova solo entità già viste nel training
- Ambiguità label: stesso text_span con più label → predice più entità (poi magari ne elimina una nei dedup)
- Overlap aggressivo: può buttare entità corrette annidate/sovrapposte

## Load Training Data and Extract Entities

In [13]:
import json
from pathlib import Path
from collections import Counter, defaultdict
PROJECT_ROOT = Path.cwd().parents[1]
DATA_ROOT = PROJECT_ROOT / "data" / "GutBrainIE_Full_Collection_2025"
ANNOTATIONS_DIR = DATA_ROOT / "Annotations"

train_files = [
    ANNOTATIONS_DIR / "Train" / "gold_quality" / "json_format" / "train_gold.json",
    ANNOTATIONS_DIR / "Train" / "platinum_quality" / "json_format" / "train_platinum.json",
    ANNOTATIONS_DIR / "Train" / "silver_quality" / "json_format" / "train_silver.json",
]


# span_norm -> Counter(labels)
label_counts_by_span = defaultdict(Counter)

for train_file in train_files:
    if not train_file.exists():
        raise FileNotFoundError(f"Missing file: {train_file}")

    with train_file.open(encoding="utf-8") as f:
        train_data = json.load(f)

    for _, article in train_data.items():
        for ent in article["entities"]:
            span = (ent["text_span"] or "").strip()
            label = ent["label"]

            if not span:
                continue

            span_norm = span.lower()
            label_counts_by_span[span_norm][label] += 1

    print(f"Loaded {train_file.name}")

def pick_majority_label(counter: Counter) -> str:
    # label più frequente; tie-break deterministico (ordine alfabetico)
    max_count = max(counter.values())
    top = sorted([lbl for lbl, c in counter.items() if c == max_count])
    return top[0]

Loaded train_gold.json
Loaded train_platinum.json
Loaded train_silver.json


## Load Dev Data

In [14]:
dev_data_path = (
    ANNOTATIONS_DIR
    / "Dev"
    / "json_format"
    / "dev.json"
)

with dev_data_path.open(encoding="utf-8") as f:
    dev_data = json.load(f)

print(f"Loaded {len(dev_data)} dev documents")


Loaded 40 dev documents


## Predict Entities on Dev Set

In [15]:
from tqdm import tqdm
import re
def find_all_occurrences(text, entity_text):
    """
    Find all occurrences of entity_text in text.
    Returns list of (start_idx, end_idx) tuples.
    """
    occurrences = []
    start = 0
    while True:
        pos = text.find(entity_text, start)
        if pos == -1:
            break
        occurrences.append((pos, pos + len(entity_text)))
        start = pos + 1
    return occurrences

def remove_overlapping_entities(entities):
    """
    Remove overlapping entities, keeping only the longest span.
    entities: list of dicts with start_idx, end_idx, text_span, label, location
    """
    if not entities:
        return []
    
    # Sort by start position, then by length (longest first)
    entities = sorted(entities, key=lambda x: (x['start_idx'], -(x['end_idx'] - x['start_idx'])))
    
    kept = []
    for entity in entities:
        # Check if this entity overlaps with any kept entity
        overlaps = False
        for kept_entity in kept:
            # Check if same location and overlapping spans
            if entity['location'] == kept_entity['location']:
                # Check for overlap
                if not (entity['end_idx'] <= kept_entity['start_idx'] or 
                       entity['start_idx'] >= kept_entity['end_idx']):
                    overlaps = True
                    break
        
        if not overlaps:
            kept.append(entity)
    
    return kept



 ### Filtri + regex token-boundary + ignorecase

In [16]:
def is_noise_span(span_norm: str, min_len: int = 3) -> bool:
    s = span_norm.strip()
    if len(s) < min_len:
        return True
    # solo punteggiatura/spazi
    if all(not ch.isalnum() for ch in s):
        return True
    return False

def compile_patterns(label_counts_by_span, min_len: int = 3):
    """
    Ritorna lista di tuple: (span_norm, compiled_regex, label)
    Usa boundary robusto: (?<!\w) ... (?!\w) per evitare match dentro parole.
    """
    patterns = []
    for span_norm, lbl_counter in label_counts_by_span.items():
        if is_noise_span(span_norm, min_len=min_len):
            continue

        label = pick_majority_label(lbl_counter)

        # boundary "token-like": non deve avere caratteri \w attaccati ai lati
        pat = re.compile(rf"(?<!\w){re.escape(span_norm)}(?!\w)", flags=re.IGNORECASE)
        patterns.append((span_norm, pat, label))

    # prima i più lunghi (riduce conflitti e accelera pruning)
    patterns.sort(key=lambda x: len(x[0]), reverse=True)
    return patterns

PATTERNS = compile_patterns(label_counts_by_span, min_len=3)
print("Total patterns after filtering:", len(PATTERNS))


Total patterns after filtering: 7666


In [17]:
def is_noise_span(span_norm: str, min_len: int = 3) -> bool:
    s = span_norm.strip()
    if len(s) < min_len:
        return True
    # solo punteggiatura/spazi
    if all(not ch.isalnum() for ch in s):
        return True
    return False

def compile_patterns(label_counts_by_span, min_len: int = 3):
    """
    Ritorna lista di tuple: (span_norm, compiled_regex, label)
    Usa boundary robusto: (?<!\w) ... (?!\w) per evitare match dentro parole.
    """
    patterns = []
    for span_norm, lbl_counter in label_counts_by_span.items():
        if is_noise_span(span_norm, min_len=min_len):
            continue

        label = pick_majority_label(lbl_counter)

        # boundary "token-like": non deve avere caratteri \w attaccati ai lati
        pat = re.compile(rf"(?<!\w){re.escape(span_norm)}(?!\w)", flags=re.IGNORECASE)
        patterns.append((span_norm, pat, label))

    # prima i più lunghi (riduce conflitti e accelera pruning)
    patterns.sort(key=lambda x: len(x[0]), reverse=True)
    return patterns

PATTERNS = compile_patterns(label_counts_by_span, min_len=3)
print("Total patterns after filtering:", len(PATTERNS))


Total patterns after filtering: 7666


## predictions

In [18]:
# Process each document in dev set
predictions = {}

for pmid, article in tqdm(dev_data.items(), desc="Processing dev data"):
    title = article["metadata"]["title"]
    abstract = article["metadata"]["abstract"]

    predicted_entities = []

    for location, text in [("title", title), ("abstract", abstract)]:
        # Per matching ignorecase usiamo anche una versione lower
        text_lower = text.lower()

        for span_norm, regex, label in PATTERNS:
            # finditer sul lower? No: regex è IGNORECASE, lo applichiamo sul testo originale
            for m in regex.finditer(text):
                start_idx = m.start()
                end_excl = m.end()

                # IMPORTANTISSIMO: salva la substring reale del testo (casing esatto)
                matched_text_span = text[start_idx:end_excl]

                predicted_entities.append({
                    "start_idx": start_idx,
                    "end_idx": end_excl - 1,  # inclusivo
                    "location": location,
                    "text_span": matched_text_span,
                    "label": label
                })

    # overlap prune (puoi anche commentarlo e lasciare che l'eval lo faccia,
    # ma tenerlo qui riduce output e rumore)
    predicted_entities = remove_overlapping_entities(predicted_entities)

    predictions[pmid] = {"entities": predicted_entities}

print(f"\nProcessed {len(predictions)} documents")
print(f"Total entities predicted: {sum(len(p['entities']) for p in predictions.values())}")

Processing dev data: 100%|██████████| 40/40 [00:19<00:00,  2.04it/s]


Processed 40 documents
Total entities predicted: 1854


## Save Predictions

In [19]:
# Save predictions to file
PREDICTIONS_DIR = PROJECT_ROOT / "src" / "predictions"
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

output_path = PREDICTIONS_DIR / "vanilla_NER.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print(f"Predictions saved to {output_path}")

Predictions saved to C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\predictions\vanilla_NER.json


## Evaluate Performance

In [20]:
# Self-contained evaluation function (adapted from official script)
# Avoids importing evaluate.py which has hardcoded paths

def remove_duplicated_entities(predictions):
    """Remove duplicated entities from predictions."""
    removed_count = 0
    for pmid in list(predictions.keys()):
        seen = set()
        deduped = []
        for ent in predictions[pmid]["entities"]:
            key = (ent["start_idx"], ent["end_idx"], ent["location"], ent["label"])
            if key not in seen:
                seen.add(key)
                deduped.append(ent)
            else:
                removed_count += 1
        predictions[pmid]["entities"] = deduped
    
    if removed_count > 0:
        print(f"Removed {removed_count} duplicated entities from predictions")

def remove_overlapping_entities_eval(predictions):
    """Remove overlapping entities, keeping longest spans."""
    removed_count = 0

    for pmid in list(predictions.keys()):
        original_len = len(predictions[pmid]['entities'])
        
        # Group entities by location
        groups = {'title': [], 'abstract': []}
        for ent in predictions[pmid]['entities']:
            loc = ent["location"]
            groups[loc].append(ent)

        # For each location, build overlap clusters and select the longest
        keepers = set()
        for loc in groups:
            group = groups[loc]
            group = sorted(group, key=lambda e: e["start_idx"])

            clusters = []
            cluster = []
            current_end = None

            for ent in group:
                if not cluster:
                    cluster = [ent]
                    current_end = ent["end_idx"]
                else:
                    if ent["start_idx"] < current_end:
                        cluster.append(ent)
                        if ent["end_idx"] > current_end:
                            current_end = ent["end_idx"]
                    else:
                        clusters.append(cluster)
                        cluster = [ent]
                        current_end = ent["end_idx"]
            if cluster:
                clusters.append(cluster)

            # Pick the longest entity in each cluster
            for clust in clusters:
                longest = clust[0]
                max_len = longest["end_idx"] - longest["start_idx"]
                for ent in clust[1:]:
                    length = ent["end_idx"] - ent["start_idx"]
                    if length > max_len:
                        longest = ent
                        max_len = length
                keepers.add((longest["start_idx"],
                             longest["end_idx"],
                             longest["location"]))

        # Rebuild the entity list
        deduped = []
        for ent in predictions[pmid]['entities']:
            key = (ent["start_idx"], ent["end_idx"], ent["location"])
            if key in keepers:
                deduped.append(ent)
                keepers.remove(key)

        predictions[pmid]["entities"] = deduped
        removed_count += (original_len - len(deduped))

    if removed_count > 0:
        print(f"Removed {removed_count} overlapping entities")

def evaluate_ner(predictions, ground_truth):
    """
    Evaluate NER predictions against ground truth.
    Based on the official evaluation script.
    """
    # Remove duplicated and overlapping entities
    remove_duplicated_entities(predictions)
    remove_overlapping_entities_eval(predictions)
    
    LEGAL_ENTITY_LABELS = [
        "anatomical location",
        "animal",
        "bacteria",
        "biomedical technique",
        "chemical",
        "DDF",
        "dietary supplement",
        "drug",
        "food",
        "gene",
        "human",
        "microbiome",
        "statistical technique"
    ]
    
    ground_truth_NER = dict()
    count_annotated_entities_per_label = {}
    
    for pmid, article in ground_truth.items():
        if pmid not in ground_truth_NER:
            ground_truth_NER[pmid] = []
        for entity in article['entities']:
            start_idx = int(entity["start_idx"])
            end_idx = int(entity["end_idx"])
            location = str(entity["location"])
            text_span = str(entity["text_span"])
            label = str(entity["label"]) 
            
            entry = (start_idx, end_idx, location, text_span, label)
            ground_truth_NER[pmid].append(entry)
            
            if label not in count_annotated_entities_per_label:
                count_annotated_entities_per_label[label] = 0
            count_annotated_entities_per_label[label] += 1

    count_predicted_entities_per_label = {label: 0 for label in list(count_annotated_entities_per_label.keys())}
    count_true_positives_per_label = {label: 0 for label in list(count_annotated_entities_per_label.keys())}

    for pmid in predictions.keys():
        entities = predictions[pmid]['entities']
        
        for entity in entities:
            start_idx = int(entity["start_idx"])
            end_idx = int(entity["end_idx"])
            location = str(entity["location"])
            text_span = str(entity["text_span"])
            label = str(entity["label"]) 
            
            if label not in LEGAL_ENTITY_LABELS:
                print(f'Warning: Illegal label {label} for entity: {entity}')
                continue

            if label in count_predicted_entities_per_label:
                count_predicted_entities_per_label[label] += 1

            entry = (start_idx, end_idx, location, text_span, label)
            if entry in ground_truth_NER[pmid]:
                count_true_positives_per_label[label] += 1

    count_annotated_entities = sum(count_annotated_entities_per_label[label] for label in list(count_annotated_entities_per_label.keys()))
    count_predicted_entities = sum(count_predicted_entities_per_label[label] for label in list(count_annotated_entities_per_label.keys()))
    count_true_positives = sum(count_true_positives_per_label[label] for label in list(count_annotated_entities_per_label.keys()))

    micro_precision = count_true_positives / (count_predicted_entities + 1e-10)
    micro_recall = count_true_positives / (count_annotated_entities + 1e-10)
    micro_f1 = 2 * ((micro_precision * micro_recall) / (micro_precision + micro_recall + 1e-10))

    precision, recall, f1 = 0, 0, 0
    n = 0
    for label in list(count_annotated_entities_per_label.keys()):
        n += 1
        current_precision = count_true_positives_per_label[label] / (count_predicted_entities_per_label[label] + 1e-10) 
        current_recall = count_true_positives_per_label[label] / (count_annotated_entities_per_label[label] + 1e-10) 
        
        precision += current_precision
        recall += current_recall
        f1 += 2 * ((current_precision * current_recall) / (current_precision + current_recall + 1e-10))
    
    precision = precision / n
    recall = recall / n
    f1 = f1 / n

    return precision, recall, f1, micro_precision, micro_recall, micro_f1

# Evaluate
precision, recall, f1, micro_precision, micro_recall, micro_f1 = evaluate_ner(predictions, dev_data)

print("="*60)
print("VANILLA NER BASELINE RESULTS")
print("="*60)
print("\nMacro-averaged Metrics:")
print(f"  Macro-Precision: {precision:.4f}")
print(f"  Macro-Recall:    {recall:.4f}")
print(f"  Macro-F1 Score:  {f1:.4f}")

print("\nMicro-averaged Metrics:")
print(f"  Micro-Precision: {micro_precision:.4f}")
print(f"  Micro-Recall:    {micro_recall:.4f}")
print(f"  Micro-F1 Score:  {micro_f1:.4f}")
print("="*60)

VANILLA NER BASELINE RESULTS

Macro-averaged Metrics:
  Macro-Precision: 0.2969
  Macro-Recall:    0.4966
  Macro-F1 Score:  0.3582

Micro-averaged Metrics:
  Micro-Precision: 0.3436
  Micro-Recall:    0.5703
  Micro-F1 Score:  0.4288


By introducing case-insensitive matching with token boundaries, majority-label disambiguation, and noise filtering on short spans, the vanilla gazetteer-based NER baseline significantly improves precision while preserving high recall.

The Micro-F1 score increases from 0.38 to 0.43, demonstrating that simple linguistic constraints can substantially reduce false positives without relying on learned models.

## Example Predictions

In [21]:
# Show example predictions
print("Example Predictions:\n")

# Get first few documents
sample_pmids = list(dev_data.keys())[:5]

for pmid in sample_pmids:
    article = dev_data[pmid]
    pred = predictions[pmid]
    
    print(f"Document PMID: {pmid}")
    print(f"Title: {article['metadata']['title'][:100]}...")
    print(f"\nGold entities: {len(article['entities'])}")
    print(f"Predicted entities: {len(pred['entities'])}")
    
    # Show first few predicted entities
    print("\nSample predictions:")
    for entity in pred['entities'][:5]:
        print(f"  - '{entity['text_span']}' [{entity['label']}] in {entity['location']}")
    
    # Calculate match statistics for this document
    gold_set = set()
    for entity in article['entities']:
        gold_set.add((
            entity['start_idx'],
            entity['end_idx'],
            entity['location'],
            entity['text_span'],
            entity['label']
        ))
    
    pred_set = set()
    for entity in pred['entities']:
        pred_set.add((
            entity['start_idx'],
            entity['end_idx'],
            entity['location'],
            entity['text_span'],
            entity['label']
        ))
    
    correct = len(gold_set & pred_set)
    missed = len(gold_set - pred_set)
    wrong = len(pred_set - gold_set)
    
    print(f"\n✓ Correct: {correct}")
    print(f"✗ Missed: {missed}")
    print(f"✗ Wrong: {wrong}")
    print("-" * 80)
    print()

Example Predictions:

Document PMID: 36532064
Title: Hypothesis of a potential BrainBiota and its relation to CNS autoimmune inflammation....

Gold entities: 19
Predicted entities: 33

Sample predictions:
  - 'agents' [drug] in abstract
  - 'CNS' [anatomical location] in title
  - 'pathogenesis' [DDF] in abstract
  - 'inflammation' [DDF] in title
  - 'neurological diseases' [DDF] in abstract

✓ Correct: 13
✗ Missed: 6
✗ Wrong: 20
--------------------------------------------------------------------------------

Document PMID: 37212075
Title: IgA-Biome Profiles Correlate with Clinical Parkinson's Disease Subtypes....

Gold entities: 21
Predicted entities: 38

Sample predictions:
  - 'Parkinson's disease' [DDF] in abstract
  - 'Clinical' [DDF] in title
  - 'neurodegenerative disorder' [DDF] in abstract
  - 'Parkinson's Disease' [DDF] in title
  - 'gut microbiome' [microbiome] in abstract

✓ Correct: 5
✗ Missed: 16
✗ Wrong: 33
---------------------------------------------------------------

## Analysis: Entity Distribution by Label

In [22]:
from collections import Counter

# Count entities by label in predictions
pred_label_counts = Counter()
for pmid, pred in predictions.items():
    for entity in pred['entities']:
        pred_label_counts[entity['label']] += 1

# Count entities by label in gold standard
gold_label_counts = Counter()
for pmid, article in dev_data.items():
    for entity in article['entities']:
        gold_label_counts[entity['label']] += 1

print("Entity Distribution by Label:")
print("="*60)
print(f"{'Label':<25} {'Gold':<10} {'Predicted':<10}")
print("-"*60)

all_labels = set(gold_label_counts.keys()) | set(pred_label_counts.keys())
for label in sorted(all_labels):
    print(f"{label:<25} {gold_label_counts[label]:<10} {pred_label_counts[label]:<10}")

print("-"*60)
print(f"{'TOTAL':<25} {sum(gold_label_counts.values()):<10} {sum(pred_label_counts.values()):<10}")

Entity Distribution by Label:
Label                     Gold       Predicted 
------------------------------------------------------------
DDF                       379        583       
anatomical location       76         214       
animal                    73         88        
bacteria                  54         97        
biomedical technique      36         50        
chemical                  131        215       
dietary supplement        27         58        
drug                      60         102       
food                      26         11        
gene                      39         63        
human                     86         145       
microbiome                127        219       
statistical technique     3          9         
------------------------------------------------------------
TOTAL                     1117       1854      
